# Lesson 8 — Unconstrained NLP

## Learning objectives

1. State first- and second-order optimality conditions.
2. Walk through gradient descent, Newton's method, and quasi-Newton (BFGS).
3. Understand line search vs trust region; appreciate why pure Newton can fail.
4. Use `discopt`'s NLP solver and inspect convergence behaviour.

## 1. Optimality conditions

For unconstrained $\min_x f(x)$ with $f \in C^2$:

- **First-order:** $\nabla f(x^\star) = 0$ (necessary).
- **Second-order:** $\nabla^2 f(x^\star) \succeq 0$ (necessary), $\succ 0$ sufficient for strict local min.

These are necessary, not sufficient — saddle points satisfy first-order. {cite:p}`Nocedal2006` Ch 2.

## 2. Gradient descent

$$x_{k+1} = x_k - \alpha_k \nabla f(x_k).$$

- Cheap (just need gradients).
- Linear convergence on strongly convex $f$ with condition number $\kappa$: error shrinks by $(1 - 1/\kappa)$ each step. Slow when $\kappa$ is large.
- Step $\alpha_k$: line search (Armijo, Wolfe) or fixed $\alpha = 1/L$ if you know the Lipschitz constant {cite:p}`Nocedal2006`.

## 3. Newton's method

$$x_{k+1} = x_k - [\nabla^2 f(x_k)]^{-1} \nabla f(x_k).$$

- **Quadratic convergence** in a neighbourhood of $x^\star$ when $\nabla^2 f(x^\star)$ is *positive definite* (not merely PSD) and the Hessian is locally Lipschitz {cite:p}`Nocedal2006` (Thm 3.5). Without the PD hypothesis, convergence can drop to linear — e.g., for $f(x) = x^4$ the Hessian vanishes at the optimum and Newton gives $x_{k+1} = \frac{2}{3} x_k$.
- Requires Hessians (and inverting them).
- Far from optimum, can diverge — modified Newton with damping or trust region {cite:p}`Nocedal2006`.

## 4. Quasi-Newton (BFGS)

Build an approximate Hessian $B_k$ from gradient differences:

$$B_{k+1} = B_k - \frac{B_k s_k s_k^\top B_k}{s_k^\top B_k s_k} + \frac{y_k y_k^\top}{y_k^\top s_k},$$

with $s_k = x_{k+1} - x_k$, $y_k = \nabla f(x_{k+1}) - \nabla f(x_k)$. Superlinear convergence; no Hessian needed; the workhorse for unconstrained NLP.

In [ ]:
# Standard discopt course imports. The lessons target the real
# `discopt.modeling.core.Model` API: `m.continuous(name, shape=..., lb=..., ub=...)`,
# `m.binary(name, shape=...)`, `m.integer(name, shape=..., lb=..., ub=...)`,
# `m.subject_to(...)`, `m.minimize(...) / .maximize(...)`, `m.solve(...)`.
# Result attributes: `r.status`, `r.objective`, `r.gap`, `r.bound`,
# `r.wall_time`, `r.node_count`, and `r.value(var)` for variable arrays.
import numpy as np
import discopt as do
import discopt.modeling as dm


In [ ]:
import numpy as np

# Rosenbrock — non-convex but a unique global min at (1,1).
# We compute gradients via JAX directly (which discopt itself uses internally).
import jax
import jax.numpy as jnp

def f(x): return (1 - x[0])**2 + 100 * (x[1] - x[0]**2)**2
grad_f = jax.grad(lambda x: f(x))

x = np.array([-1.2, 1.0])
for k in range(50):
    g = np.asarray(grad_f(jnp.array(x)))
    x = x - 0.001 * g
print("after 50 GD steps: x =", x, "  f =", f(x))
# (Far from the optimum (1,1) — GD is slow on Rosenbrock.)


## 5. Line search

We want $\alpha_k$ such that $f(x_k + \alpha_k p_k)$ decreases sufficiently. Wolfe conditions:

- $f(x + \alpha p) \le f(x) + c_1 \alpha \nabla f^\top p$ (Armijo / sufficient decrease).
- $|\nabla f(x + \alpha p)^\top p| \le c_2 |\nabla f^\top p|$ (curvature).

Practical choice $c_1 = 10^{-4}, c_2 = 0.9$. See {cite:p}`Nocedal2006` Ch 3.

## 6. Trust region

Solve a *local quadratic model* in a ball $\|p\| \le \Delta$:

$$\min_p \; m_k(p) = f_k + \nabla f_k^\top p + \tfrac{1}{2} p^\top B_k p \;\;\text{s.t.}\;\; \|p\| \le \Delta.$$

Adapt $\Delta$ based on agreement between $m_k$ and $f$. More robust than line search when Hessian has indefinite directions.

## References
{cite:p}`Nocedal2006` is the standard reference. Also {cite:p}`Wright1997`.